In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from datasets import load_dataset
from copy import deepcopy
import numpy as np
from scipy.stats import ttest_ind
import torch
load_dotenv()

True

In [4]:
df = pd.read_csv('../QA_datasets_classified_qa_eval/output_csv/frames_UND_gpt4o_Ragas.csv')
ragas_col = df['ragas_AA_short'].tolist()
pre_ragas = load_dataset("json", 
                        data_files="../QA_datasets_classified_qa_eval/intermediate/BASELINE_frames_UND_qa_gpt_with_squad_scores.jsonl",
                        split="all")
assert len(pre_ragas) == len(ragas_col), "Fatal Error: length mismatch"

Generating train split: 440 examples [00:00, 47808.24 examples/s]


In [5]:
with_ragas = pre_ragas.add_column('ragas_AA_short', ragas_col)
with_ragas.to_json("./intermediate/frames_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 72.46ba/s]


1946220

## Rewriting with Gemini
GPT-4o rewriting, then GPT-4o QA later

In [6]:
from helper_functions_qr import modification_in_batch

In [7]:
client = OpenAI(
    api_key=os.environ.get('GOOGLE_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
model = "gemini-2.5-flash"
input_file = "./intermediate/frames_UND_gpt4o_Ragas.jsonl"
output_file = "./intermediate/MODIFIED_frames_UND_gpt4o_Ragas.jsonl"



In [8]:
# 清空输出文件（如果存在）
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Cleared existing output file: {output_file}")

# 处理所有样本（按批次）
question_modification = modification_in_batch(input_file, output_file, 'answer', client, model)

Total samples to process: 440
Batch size: 3


Processing batches:   3%|▎         | 5/147 [08:02<5:41:41, 144.38s/it]

Error processing sample 16: Invalid \escape: line 3 column 376 (char 652)


Processing batches:  20%|██        | 30/147 [55:08<6:18:52, 194.29s/it]

Error processing sample 92: Invalid \escape: line 3 column 95 (char 169)


...

Error processing sample 158: Invalid \escape: line 3 column 112 (char 330)


Processing batches:  54%|█████▎    | 79/147 [2:00:26<1:18:44, 69.47s/it] 

Error processing sample 238: Invalid \escape: line 3 column 294 (char 443)


Processing batches: 100%|██████████| 147/147 [3:55:35<00:00, 96.16s/it]   


All batch processing completed! Total processed: 440 samples
Results saved to: ./intermediate/MODIFIED_frames_UND_gpt4o_Ragas.jsonl


In [9]:
df_view = pd.DataFrame(question_modification)
df_view

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA
0,How many years earlier would Punxsutawney Phil...,How many years earlier than his first official...,87,[88 years earlier],The query hinges on two critical pieces of inf...,0.000000,0,0.0
1,"As of August 1, 2024, which country were holde...",What country held the FIFA World Cup title at ...,France,[France],The query contains two distinct temporal refer...,1.000000,1,1.0
2,What is the name of the vocalist from the firs...,Considering Nuclear Blast Records produced Dis...,Jens Kidman,[Roy Khan],"The query is complex and layered, requiring mu...",0.000000,0,0.0
3,I have an element in mind and would like you t...,I have an element in mind and would like you t...,Mendelevium is named after Dmitri Mendeleev.,[Niels Bohr],"To solve this, first identify the scientist wh...",0.000000,0,0.0
4,"As of Aug 3, 2024, the artist who released the...","As of Aug 3, 2024, DJ Khaled, the artist who r...",2,[Four],The query links two entities (the artist of 'F...,0.000000,0,0.0
...,...,...,...,...,...,...,...,...
435,What is the name of the father of the first co...,What is the name of the brother of one of the ...,Peter Treschow,[Philip Hansteen],The query involves tracing a complex genealogi...,0.000000,0,0.0
436,Who was the winner of Tour de France the same ...,Who was the winner of the Tour de France in th...,Louison Bobet,[Louison Bobet],The query requires identifying a specific year...,1.000000,1,1.0
437,A 2002 science fiction novel by an American au...,The 2002 science fiction novel 'The House of t...,The Sea of Trolls trilogy,[The Xenogenesis Trilogy],The query provides specific details about the ...,0.333333,0,0.0
438,Which movie musical produced a song that was i...,Which movie musical produced a song inspired b...,Fame,[Cats],The query requires identifying a movie musical...,0.000000,0,0.0


## Modified queries QA using GPT-4o

### Loading modified data

In [10]:
modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_frames_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)


#modified_set = modified_set.remove_columns(["model_new_answer"])

#modified_set.to_json(
    #"./intermediate/MODIFIED_frames_UND_gpt4o_Ragas.jsonl",
    #orient="records",
    #lines=True
#)

Generating train split: 440 examples [00:00, 70427.94 examples/s]


### Implementation

In [11]:
from helper_functions_qr import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)
client = OpenAI(
    api_key=os.environ.get('OPENAI_API_KEY')
)

In [12]:
modified_results = batch_QA_with_progress(
    modified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_new_answer",
    fill_value=["error"],
    client=client,
    model="gpt-4o-2024-11-20",
    temperature=0.0
)

Running model_new_answer: 100%|██████████| 44/44 [07:43<00:00, 10.52s/it]


In [13]:
qa_modified = deepcopy(modified_set)
for key in modified_results:
    qa_modified = qa_modified.add_column(key, modified_results[key])

qa_modified.to_json("./intermediate/MODIFIED_frames_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)
df_qa_modified = pd.read_json("./intermediate/MODIFIED_frames_UND_gpt4o_Ragas.jsonl", lines=True)
#df_qa_modified.to_csv('produced_files/modification_pilot_qa.csv')
df_qa_modified

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 153.78ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer
0,How many years earlier would Punxsutawney Phil...,How many years earlier than his first official...,87,[88 years earlier],The query hinges on two critical pieces of inf...,0.000000,0,0.0,[- 87 years earlier]
1,"As of August 1, 2024, which country were holde...",What country held the FIFA World Cup title at ...,France,[France],The query contains two distinct temporal refer...,1.000000,1,1.0,[France]
2,What is the name of the vocalist from the firs...,Considering Nuclear Blast Records produced Dis...,Jens Kidman,[Roy Khan],"The query is complex and layered, requiring mu...",0.000000,0,0.0,[Jens Kidman]
3,I have an element in mind and would like you t...,I have an element in mind and would like you t...,Mendelevium is named after Dmitri Mendeleev.,[Niels Bohr],"To solve this, first identify the scientist wh...",0.000000,0,0.0,[Dmitri Mendeleev]
4,"As of Aug 3, 2024, the artist who released the...","As of Aug 3, 2024, DJ Khaled, the artist who r...",2,[Four],The query links two entities (the artist of 'F...,0.000000,0,0.0,[1]
...,...,...,...,...,...,...,...,...,...
435,What is the name of the father of the first co...,What is the name of the brother of one of the ...,Peter Treschow,[Philip Hansteen],The query involves tracing a complex genealogi...,0.000000,0,0.0,[Hans Steenbuch]
436,Who was the winner of Tour de France the same ...,Who was the winner of the Tour de France in th...,Louison Bobet,[Louison Bobet],The query requires identifying a specific year...,1.000000,1,1.0,[Louison Bobet]
437,A 2002 science fiction novel by an American au...,The 2002 science fiction novel 'The House of t...,The Sea of Trolls trilogy,[The Xenogenesis Trilogy],The query provides specific details about the ...,0.333333,0,0.0,[The Sea of Trolls Trilogy]
438,Which movie musical produced a song that was i...,Which movie musical produced a song inspired b...,Fame,[Cats],The query requires identifying a movie musical...,0.000000,0,0.0,[The Sound of Music]


## Evaluations

### Squad EM+F1

In [14]:
# Helper functions updated, RESTART!!
from helper_functions_qr import evaluate_squad_per_sample_multi_ref_pred

modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_frames_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

qa_modified = deepcopy(modified_set)

Generating train split: 440 examples [00:00, 95969.51 examples/s]


In [15]:
squad_scored_modified, modified_f1_list, modified_em_list = evaluate_squad_per_sample_multi_ref_pred(qa_modified)
squad_scored_modified.to_json("./intermediate/MODIFIED_Gemini_frames_UND_gpt4o_new_squad.jsonl", orient="records", lines=True)

df = pd.read_json("./intermediate/MODIFIED_Gemini_frames_UND_gpt4o_new_squad.jsonl", lines=True)
df

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 196.47ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1
0,How many years earlier would Punxsutawney Phil...,How many years earlier than his first official...,87,[88 years earlier],The query hinges on two critical pieces of inf...,0.000000,0,0.0,[- 87 years earlier],0,0.5
1,"As of August 1, 2024, which country were holde...",What country held the FIFA World Cup title at ...,France,[France],The query contains two distinct temporal refer...,1.000000,1,1.0,[France],1,1.0
2,What is the name of the vocalist from the firs...,Considering Nuclear Blast Records produced Dis...,Jens Kidman,[Roy Khan],"The query is complex and layered, requiring mu...",0.000000,0,0.0,[Jens Kidman],1,1.0
3,I have an element in mind and would like you t...,I have an element in mind and would like you t...,Mendelevium is named after Dmitri Mendeleev.,[Niels Bohr],"To solve this, first identify the scientist wh...",0.000000,0,0.0,[Dmitri Mendeleev],0,0.5
4,"As of Aug 3, 2024, the artist who released the...","As of Aug 3, 2024, DJ Khaled, the artist who r...",2,[Four],The query links two entities (the artist of 'F...,0.000000,0,0.0,[1],0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
435,What is the name of the father of the first co...,What is the name of the brother of one of the ...,Peter Treschow,[Philip Hansteen],The query involves tracing a complex genealogi...,0.000000,0,0.0,[Hans Steenbuch],0,0.0
436,Who was the winner of Tour de France the same ...,Who was the winner of the Tour de France in th...,Louison Bobet,[Louison Bobet],The query requires identifying a specific year...,1.000000,1,1.0,[Louison Bobet],1,1.0
437,A 2002 science fiction novel by an American au...,The 2002 science fiction novel 'The House of t...,The Sea of Trolls trilogy,[The Xenogenesis Trilogy],The query provides specific details about the ...,0.333333,0,0.0,[The Sea of Trolls Trilogy],1,1.0
438,Which movie musical produced a song that was i...,Which movie musical produced a song inspired b...,Fame,[Cats],The query requires identifying a movie musical...,0.000000,0,0.0,[The Sound of Music],0,0.0


In [16]:
modified_mean_em = np.mean(modified_em_list)  # em_scores: EM list per sample
modified_mean_f1 = np.mean(modified_f1_list)  # f1_scores F1 list per sample
print(f"New answers after modification Exact Match (avg): {modified_mean_em * 100:.2f}")
print(f"New answers after modification F1 Score (avg): {modified_mean_f1 * 100:.2f}")

original_em_list = qa_modified['original_em']
original_f1_list = qa_modified['original_f1']

original_mean_em = np.mean(original_em_list)  # em_scores: EM list per sample
original_mean_f1 = np.mean(original_f1_list)  # f1_scores F1 list per sample
print(f"Original answers Exact Match (avg): {original_mean_em * 100:.2f}")
print(f"Original answers F1 Score (avg): {original_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(modified_f1_list, original_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(modified_em_list, original_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

New answers after modification Exact Match (avg): 23.41
New answers after modification F1 Score (avg): 41.59
Original answers Exact Match (avg): 12.95
Original answers F1 Score (avg): 24.43
F1: t=6.831, p=0.0000
EM: t=4.053, p=0.0001


### Ragas AA

In [17]:
from helper_functions_qr import answer_accuracy_modified
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

In [18]:
squad_scored_modified = load_dataset("json",
    data_files="./intermediate/MODIFIED_Gemini_frames_UND_gpt4o_new_squad.jsonl",
    split="train")
result_with_AA = await answer_accuracy_modified(squad_scored_modified, evaluator_llm)
result_with_AA.to_csv("./output_csv/MODIFIED_Gemini_frames_UND_gpt4o_all_new_scores.csv")

Generating train split: 440 examples [00:00, 14118.02 examples/s]
Calculating short answer accuracy:   3%|▎         | 13/440 [00:36<19:32,  2.75s/it]

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 18.32ba/s]


545915

In [19]:
original_AA = list(result_with_AA["original_AA"])
modified_AA = list(result_with_AA["new_AA"])

original_mean_AA = np.mean(original_AA)
print(f"original AA (avg): {original_mean_AA * 100:.2f}")


modified_mean_AA = np.mean(modified_AA)
print(f"modified AA (avg): {modified_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(modified_AA, original_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

original AA (avg): 27.16
modified AA (avg): 51.53
AA: t=7.941, p=0.0000


## Re-Classification

In [2]:
from helper_functions_qr import (
    batch_generate_responses_qwen3,
    get_judgments_from_responses,
    run_experiment,
    prepare_test_prompts)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


### Loading data

In [3]:
reclassify_file = "./output_csv/MODIFIED_Gemini_frames_UND_gpt4o_all_new_scores.csv"
df_reclassify_file = pd.read_csv(reclassify_file)

### Loading model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
Qwen3_4B = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)
# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:12<00:00,  4.21s/it]


cuda


### Prepare prompts

In [5]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [6]:
test_prompts = prepare_test_prompts(df_reclassify_file, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 440
Generation complete: 440 prompts
Average prompt length: 636 bytes (~159 tokens)

Analyze the following input user query:

{"query": "How many years earlier than his first official prediction in 1887 would Punxsutawney Phil have to be canonically alive to have made a Groundhog Day prediction in the federal district where the U.S. Capitol was first occupied by Congress?"}

Please provide your analysis in the following JSON format:

{"query": "How many years earlier than his first official prediction in 1887 would Punxsutawney Phil have to be canonically alive to have made a Groundhog Day prediction in the federal district where the U.S. Capitol was first occupied by Congress?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [7]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df_reclassify_file)
test_df

100%|██████████| 88/88 [1:34:18<00:00, 64.30s/it] 


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1,new_AA,MODIFIED_thinking,MODIFIED_model_response,MODIFIED_model_pred
0,How many years earlier would Punxsutawney Phil...,How many years earlier than his first official...,87,['88 years earlier'],The query hinges on two critical pieces of inf...,0.000000,0,0.0,['- 87 years earlier'],0,0.5,1.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""How many years earlier than his...",fully specified
1,"As of August 1, 2024, which country were holde...",What country held the FIFA World Cup title at ...,France,['France'],The query contains two distinct temporal refer...,1.000000,1,1.0,['France'],1,1.0,1.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What country held the FIFA Wo...",fully specified
2,What is the name of the vocalist from the firs...,Considering Nuclear Blast Records produced Dis...,Jens Kidman,['Roy Khan'],"The query is complex and layered, requiring mu...",0.000000,0,0.0,['Jens Kidman'],1,1.0,1.0,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""Considering Nuclear Blast Recor...",underspecified
3,I have an element in mind and would like you t...,I have an element in mind and would like you t...,Mendelevium is named after Dmitri Mendeleev.,['Niels Bohr'],"To solve this, first identify the scientist wh...",0.000000,0,0.0,['Dmitri Mendeleev'],0,0.5,1.0,"<think>\nOkay, let me try to figure this out. ...","{\n ""query"": ""I have an element in mind and w...",fully specified
4,"As of Aug 3, 2024, the artist who released the...","As of Aug 3, 2024, DJ Khaled, the artist who r...",2,['Four'],The query links two entities (the artist of 'F...,0.000000,0,0.0,['1'],0,0.0,0.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""As of Aug 3, 2024, DJ Khaled, t...",underspecified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
435,What is the name of the father of the first co...,What is the name of the brother of one of the ...,Peter Treschow,['Philip Hansteen'],The query involves tracing a complex genealogi...,0.000000,0,0.0,['Hans Steenbuch'],0,0.0,0.0,"<think>\nOkay, let's see. The user is asking f...","{\n ""query"": ""What is the name of the broth...",fully specified
436,Who was the winner of Tour de France the same ...,Who was the winner of the Tour de France in th...,Louison Bobet,['Louison Bobet'],The query requires identifying a specific year...,1.000000,1,1.0,['Louison Bobet'],1,1.0,1.0,"<think>\nOkay, let's see. The user is asking f...","{\n ""query"": ""Who was the winner of the Tour ...",fully specified
437,A 2002 science fiction novel by an American au...,The 2002 science fiction novel 'The House of t...,The Sea of Trolls trilogy,['The Xenogenesis Trilogy'],The query provides specific details about the ...,0.333333,0,0.0,['The Sea of Trolls Trilogy'],1,1.0,1.0,"<think>\nOkay, let's break down this query ste...","{\n ""query"": ""The 2002 science fiction novel ...",fully specified
438,Which movie musical produced a song that was i...,Which movie musical produced a song inspired b...,Fame,['Cats'],The query requires identifying a movie musical...,0.000000,0,0.0,['The Sound of Music'],0,0.0,0.0,"<think>\nOkay, let's see. The user is asking w...","{\n ""query"": ""Which movie musical produced a ...",fully specified


In [8]:
test_df['MODIFIED_model_pred'].value_counts(normalize=True)


MODIFIED_model_pred
fully specified    0.702273
underspecified     0.297727
Name: proportion, dtype: float64

In [9]:
test_df['MODIFIED_model_pred'].value_counts()

MODIFIED_model_pred
fully specified    309
underspecified     131
Name: count, dtype: int64

In [10]:
test_df.to_csv('./output_csv/frames_UND_Gemini_rewritten_reclassified.csv')

## Checking Leakage - Lexical Overlap Analysis

In [ ]:
import re
from collections import Counter
import pandas as pd
from helper_functions_qr import (tokenize, get_ngrams, jaccard_similarity,ngram_overlap_f1,compute_metrics)

In [5]:
df_analysis = pd.read_csv('./output_csv/frames_UND_Gemini_rewritten_reclassified.csv')

In [6]:
# Original Q-golden A pair
orig_metrics = df_analysis.apply(
    lambda row: compute_metrics(row["original_question"], row["short_answer"]),
    axis=1, result_type="expand"
).add_prefix("orig_")

# Rewritten Q-golden A pair
rewr_metrics = df_analysis.apply(
    lambda row: compute_metrics(row["modified_question"], row["short_answer"]),
    axis=1, result_type="expand"
).add_prefix("rewr_")

results = pd.concat([df_analysis, orig_metrics, rewr_metrics], axis=1)

metrics = ["jaccard", "unigram_f1", "bigram_f1"]
summary = pd.DataFrame({
    "original":  results[[f"orig_{m}" for m in metrics]].mean().values,
    "rewritten": results[[f"rewr_{m}" for m in metrics]].mean().values,
}, index=metrics)
summary["delta"] = summary["rewritten"] - summary["original"]

print(summary.round(4))

            original  rewritten   delta
jaccard       0.0506     0.0656  0.0151
unigram_f1    0.0725     0.0912  0.0187
bigram_f1     0.0312     0.0475  0.0164


In [7]:
from scipy import stats

print("\nWilcoxon signed-rank test (paired, two-sided):")
for m in metrics:
    stat, p = stats.wilcoxon(results[f"orig_{m}"], results[f"rewr_{m}"])
    print(f"  {m:15s}  statistic={stat:.1f}  p={p:.4f}")


Wilcoxon signed-rank test (paired, two-sided):
  jaccard          statistic=5276.5  p=0.0000
  unigram_f1       statistic=5694.0  p=0.0000
  bigram_f1        statistic=783.0  p=0.0000


In [8]:
def cohens_d(a, b):
    diff = a - b
    return diff.mean() / diff.std()

for m in metrics:
    d = cohens_d(results[f"rewr_{m}"], results[f"orig_{m}"])
    print(f"{m:15s}  Cohen's d = {d:.4f}")

jaccard          Cohen's d = 0.2668
unigram_f1       Cohen's d = 0.2707
bigram_f1        Cohen's d = 0.2601


In [16]:
# Original Q - original model A pair
orig_metrics = df_analysis.apply(
    lambda row: compute_metrics(row["original_question"], row["model_original_answer"]),
    axis=1, result_type="expand"
).add_prefix("orig_")

# Rewritten Q - rewritten model A pair
rewr_metrics = df_analysis.apply(
    lambda row: compute_metrics(row["modified_question"], row["model_new_answer"]),
    axis=1, result_type="expand"
).add_prefix("rewr_")

results = pd.concat([df_analysis, orig_metrics, rewr_metrics], axis=1)

metrics = ["jaccard", "unigram_f1", "bigram_f1"]
summary = pd.DataFrame({
    "original":  results[[f"orig_{m}" for m in metrics]].mean().values,
    "rewritten": results[[f"rewr_{m}" for m in metrics]].mean().values,
}, index=metrics)
summary["delta"] = summary["rewritten"] - summary["original"]

print(summary.round(4))

            original  rewritten   delta
jaccard       0.1213     0.1006 -0.0207
unigram_f1    0.1650     0.1395 -0.0255
bigram_f1     0.0884     0.0723 -0.0161


In [17]:
print("\nWilcoxon signed-rank test (paired, two-sided):")
for m in metrics:
    stat, p = stats.wilcoxon(results[f"orig_{m}"], results[f"rewr_{m}"])
    print(f"  {m:15s}  statistic={stat:.1f}  p={p:.4f}")


Wilcoxon signed-rank test (paired, two-sided):
  jaccard          statistic=15767.5  p=0.0757
  unigram_f1       statistic=15375.5  p=0.0466
  bigram_f1        statistic=8121.0  p=0.1107
